# Attention 직접 구현 디버거 - Shape-Safe Attention Function

- Tutorial ID: `mod-attention-from-scratch-lab`
- Tutorial: Attention 직접 구현 디버거
- Section ID: `scratch-1`
- Section: Shape-Safe Attention Function


In [ ]:
# ============================================================
# 코드 읽는 법 — Shape-Safe Attention Function
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
def attention(Q,K,V,mask=None):
    assert Q.ndim == K.ndim == V.ndim == 2
    assert Q.shape[1] == K.shape[1]
    assert K.shape[0] == V.shape[0]
        scores = Q @ K.T / np.sqrt(Q.shape[1])
    if mask is not None:
                assert mask.shape == scores.shape
                scores = scores + mask
        scores = scores - scores.max(axis=-1, keepdims=True)
        A = np.exp(scores); A = A/A.sum(axis=-1, keepdims=True)
    return A @ V, A
np.random.seed(5)
Q=np.random.randn(4,3); K=np.random.randn(4,3); V=np.random.randn(4,2)
mask=np.triu(np.full((4,4), -1e9), 1)
Y,A=attention(Q,K,V,mask)
print('A rows sum:', A.sum(axis=1))
print('future mass:', np.triu(A,1).sum())
print('Y shape:', Y.shape)